In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('cleaned_titanic_data.csv')
df.head()


,Survived,Pclass,Sex,Age,Fare,Embarked,nobel,family_size,is_alone
0,0,3,0,22.0,2.110213,0,0,1.098612,0
1,1,1,1,38.0,4.280593,1,0,1.098612,0
2,1,3,1,26.0,2.188856,0,0,0.693147,1
3,1,1,1,35.0,3.990834,0,0,1.098612,0
4,0,3,0,35.0,2.202765,0,0,0.693147,1


In [3]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=42)

In [25]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def evaluate_model(y_test, y_pred):
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print(f"\nAccuracy Score: {accuracy_score(y_test, y_pred) * 100:.4f}")


In [46]:
# uses lbfgs optimizer as the dataset is small and using Hessian matrix would help in find the optimal solution faster. Also, the dataset is not too sparse so lbfgs is a good choice.  
reg = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs', l1_ratio=0)
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

In [47]:
# for logistic regression
evaluate_model(y_test, y_pred)

Confusion Matrix:
[[80 19]
 [14 52]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.81      0.83        99
           1       0.73      0.79      0.76        66

    accuracy                           0.80       165
   macro avg       0.79      0.80      0.79       165
weighted avg       0.80      0.80      0.80       165


Accuracy Score: 80.0000


In [ ]:
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid={'n_estimators': [50, 100, 200], 
        'max_depth': [None, 10, 20], 'max_leaf_nodes': [None, 5, 10, 20]}, cv=5)

result = grid.fit(X_train, y_train)


In [60]:
best_model = result.best_estimator_
y_pred_rf = best_model.predict(X_test)

print(f"Best Parameters: {result.best_params_}")
print(f"Best Cross-validation Score: {result.best_score_:.4f}\n")

evaluate_model(y_test, y_pred_rf)

Best Parameters: {'max_depth': None, 'max_leaf_nodes': 10, 'n_estimators': 50}
Best Cross-validation Score: 0.8182

Confusion Matrix:
[[93  6]
 [24 42]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.94      0.86        99
           1       0.88      0.64      0.74        66

    accuracy                           0.82       165
   macro avg       0.83      0.79      0.80       165
weighted avg       0.83      0.82      0.81       165


Accuracy Score: 81.8182


In [ ]:
# trying the experiment on the uncleaned dataset
data = pd.read_csv('.csv')

## Justification from the result
- The Decision Tree achieved the highest overall accuracy (81.82%).
- The Decision Tree had higher precision, making its positive predictions more reliable.
- Logistic Regression had better recall for Class 1, meaning it detected more positive cases.
- If the goal is overall accuracy and fewer false positives, the Decision Tree is the better choice.
- If the goal is detecting as many positive cases as possible (such as finding all survived people even if saying a few wrong is not a problem), Logistic Regression may be preferred despite its slightly lower accuracy.
- The reason why Decision tree outperforms logistic regression is the presence of some sparse data (PClass, Embarked, alone, noble) which doesn't work well with logistic regression (can perform equally with SAGA as it induce some mathematical noise i.e random choice of path)